# Pertemuan 3: Logistic Regression

[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dafx7/COMPUTING-AI-ML-OPTIMIZATION/blob/main/week3/notebooks/01_logistic_regression.ipynb)

Kita membangun logistic regression dari nol: model $p = \sigma(w\,x + a)$, loss binary cross-entropy, gradiennya, lalu melatihnya dengan gradient descent seperti Pekan 2. Sel pertama berisi data, grafik, dan pemeriksa: **jalankan saja**.

Ganti setiap `...` dengan kodemu, lalu jalankan sel cek di bawahnya.

In [ ]:
# ===== Fungsi bantu: jalankan saja, tidak perlu dibaca =====
import numpy as np
import matplotlib.pyplot as plt
np.seterr(all="ignore")  # tanpa pesan peringatan untuk exp yang sangat besar

# Data 1: jam belajar (x) dan hasil ujian (y: 1 lulus, 0 tidak lulus) untuk 20 mahasiswa (data karangan)
x = np.arange(1, 21) * 0.5
y = np.array([0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1], dtype=float)

# Data 2 (bagian D): jam belajar (x1), jam tidur malam sebelum ujian (x2), dan hasil (y2) untuk 40 mahasiswa
x1 = np.array([3.0, 3.3, 8.2, 1.4, 6.2, 7.4, 2.3, 1.0, 3.1, 6.7, 5.8, 1.9, 4.6, 6.9, 4.5, 6.5, 9.7, 7.0, 4.2, 2.3,
               3.8, 5.4, 9.0, 7.9, 3.5, 9.3, 5.0, 7.1, 1.5, 1.5, 2.4, 8.9, 7.0, 8.6, 6.6, 4.4, 5.4, 6.1, 8.7, 4.7])
x2 = np.array([8.4, 6.7, 8.0, 6.0, 7.2, 5.0, 6.1, 4.3, 3.6, 3.2, 7.2, 5.7, 8.4, 8.0, 5.3, 8.8, 6.6, 7.6, 5.4, 4.2,
               4.0, 4.1, 6.6, 3.7, 3.1, 8.0, 3.6, 5.7, 5.9, 6.7, 6.0, 8.6, 7.5, 6.4, 6.7, 6.0, 8.8, 4.4, 7.1, 6.3])
y2 = np.array([0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0,
               0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1], dtype=float)


def cek(syarat, pesan="Belum tepat."):
    try:
        benar = bool(syarat())
    except Exception as e:
        benar, pesan = False, f"{pesan} (Kode belum lengkap atau menimbulkan error: {type(e).__name__}: {e})"
    print("Benar." if benar else "Belum tepat. " + pesan)


def dekat(a, b, tol=1e-6):
    return abs(a - b) <= tol * max(1.0, abs(b))


def _sig(z):
    return 1 / (1 + np.exp(-z))


def _bce_p(p, ys):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(np.mean(-(ys * np.log(p) + (1 - ys) * np.log(1 - p))))


def akurasi(p, ys):
    """Bagian prediksi yang benar, dengan prediksi lulus kalau p >= 0,5."""
    return float(np.mean((np.asarray(p) >= 0.5) == ys))


def plot_data(xs=None, ys=None):
    xs = x if xs is None else xs; ys = y if ys is None else ys
    plt.figure(figsize=(7, 3.5))
    plt.scatter(xs[ys == 1], ys[ys == 1], s=60, color="tab:green", label="lulus (1)", zorder=3)
    plt.scatter(xs[ys == 0], ys[ys == 0], s=60, color="tab:red", label="tidak lulus (0)", zorder=3)
    plt.xlabel("jam belajar"); plt.ylabel("y"); plt.yticks([0, 1]); plt.legend(); plt.grid(alpha=0.3); plt.show()


def plot_kurva(w, a, xs=None, ys=None):
    """Data, kurva p = sigmoid(w x + a), dan decision boundary x = -a/w."""
    xs = x if xs is None else xs; ys = y if ys is None else ys
    t = np.linspace(0, xs.max() + 0.5, 300)
    plt.figure(figsize=(7, 4))
    plt.scatter(xs[ys == 1], ys[ys == 1], s=60, color="tab:green", label="lulus (1)", zorder=3)
    plt.scatter(xs[ys == 0], ys[ys == 0], s=60, color="tab:red", label="tidak lulus (0)", zorder=3)
    plt.plot(t, _sig(w * t + a), color="tab:orange", lw=2.5, label=f"p: w = {w:.3f}, a = {a:.3f}")
    if w != 0 and 0 <= -a / w <= t.max():
        plt.axvline(-a / w, color="gray", ls="--", label=f"boundary: {-a / w:.2f} jam")
    plt.xlabel("jam belajar"); plt.ylabel("p (probabilitas lulus)"); plt.legend(loc="center right"); plt.grid(alpha=0.3); plt.show()


def plot_loss(riwayat):
    """riwayat: daftar tuple yang elemen terakhirnya BCE, atau dict {label: riwayat}."""
    kumpulan = riwayat if isinstance(riwayat, dict) else {"BCE": riwayat}
    plt.figure(figsize=(7, 4))
    for label, r in kumpulan.items():
        plt.plot([t[-1] for t in r], lw=2, label=str(label))
    plt.xlabel("langkah"); plt.ylabel("BCE"); plt.legend(); plt.grid(alpha=0.3); plt.show()


def cek_gradien(fungsi, xs=None, ys=None):
    """Bandingkan gradienmu dengan perkiraan numerik di 5 titik acak."""
    xs = x if xs is None else xs; ys = y if ys is None else ys
    rng = np.random.default_rng(0); h = 1e-6
    L = lambda w, a: _bce_p(_sig(w * xs + a), ys)
    try:
        for _ in range(5):
            w, a = rng.uniform(-2, 2), rng.uniform(-6, 6)
            gw, ga = fungsi(w, a, xs, ys)
            gw_n = (L(w + h, a) - L(w - h, a)) / (2 * h)
            ga_n = (L(w, a + h) - L(w, a - h)) / (2 * h)
            if not (dekat(gw, gw_n, 1e-4) and dekat(ga, ga_n, 1e-4)):
                print(f"Belum tepat. Di w = {w:.2f}, a = {a:.2f}: gradienmu ({gw:.4f}, {ga:.4f}), perkiraan numerik ({gw_n:.4f}, {ga_n:.4f}).")
                return
        print("Benar. Gradienmu cocok dengan perkiraan numerik di 5 titik acak.")
    except Exception as e:
        print(f"Belum tepat. Kode belum lengkap atau menimbulkan error: {type(e).__name__}: {e}")


def cek_gradien_2d(fungsi):
    rng = np.random.default_rng(1); h = 1e-6
    L = lambda w1, w2, a: _bce_p(_sig(w1 * x1 + w2 * x2 + a), y2)
    try:
        for _ in range(5):
            w1, w2, a = rng.uniform(-1, 1), rng.uniform(-1, 1), rng.uniform(-8, 2)
            g = fungsi(w1, w2, a)
            gn = ((L(w1 + h, w2, a) - L(w1 - h, w2, a)) / (2 * h), (L(w1, w2 + h, a) - L(w1, w2 - h, a)) / (2 * h),
                  (L(w1, w2, a + h) - L(w1, w2, a - h)) / (2 * h))
            if not all(dekat(u, v, 1e-4) for u, v in zip(g, gn)):
                print(f"Belum tepat. Gradienmu {tuple(round(float(u), 4) for u in g)}, perkiraan numerik {tuple(round(v, 4) for v in gn)}.")
                return
        print("Benar. Ketiga gradien cocok dengan perkiraan numerik.")
    except Exception as e:
        print(f"Belum tepat. Kode belum lengkap atau menimbulkan error: {type(e).__name__}: {e}")


def plot_sweep(hasil):
    """hasil: dict {eta: riwayat}. Menggambar BCE akhir terhadap eta dan mengembalikan eta terbaik."""
    etas = sorted(hasil)
    akhir = np.array([hasil[e][-1][-1] for e in etas], dtype=float)
    print(f"{'eta':>8}  {'BCE akhir':>10}")
    for e, m in zip(etas, akhir):
        print(f"{e:>8g}  {m:>10.4f}")
    plt.figure(figsize=(7, 4))
    plt.plot(etas, akhir, "-o", color="tab:orange", lw=2)
    plt.xscale("log"); plt.xlabel("eta (skala log)"); plt.ylabel("BCE akhir"); plt.grid(alpha=0.3); plt.show()
    terbaik = etas[int(np.nanargmin(akhir))]
    print(f"eta terbaik di daftar ini: {terbaik}")
    return terbaik


def plot_boundary_2d(w1, w2, a):
    G1, G2 = np.meshgrid(np.linspace(0, 10.5, 200), np.linspace(2.5, 9.5, 200))
    plt.figure(figsize=(7, 5))
    plt.contourf(G1, G2, _sig(w1 * G1 + w2 * G2 + a), levels=np.linspace(0, 1, 11), cmap="RdYlGn", alpha=0.35)
    plt.scatter(x1[y2 == 1], x2[y2 == 1], s=50, color="tab:green", edgecolor="k", label="lulus")
    plt.scatter(x1[y2 == 0], x2[y2 == 0], s=50, color="tab:red", edgecolor="k", label="tidak lulus")
    if w2 != 0:
        t = np.linspace(0, 10.5, 50); plt.plot(t, -(w1 * t + a) / w2, color="k", lw=2, label="decision boundary p = 0,5")
    plt.xlim(0, 10.5); plt.ylim(2.5, 9.5); plt.xlabel("jam belajar (x1)"); plt.ylabel("jam tidur (x2)"); plt.legend(loc="upper left"); plt.show()

print("Fungsi bantu siap.")

In [ ]:
plot_data()

## Bagian A: model dan loss

### A1. Sigmoid

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

`np.exp(-z)` bekerja untuk satu angka maupun array.

In [ ]:
def sigmoid(z):
    return ...

In [ ]:
cek(lambda: dekat(sigmoid(0), 0.5) and sigmoid(10) > 0.9999 and sigmoid(-10) < 0.0001, 'sigmoid(0) seharusnya 0,5; sigmoid(10) hampir 1; sigmoid(-10) hampir 0.')

### A2. Prediksi

$p_i = \sigma(w\,x_i + a)$ untuk semua mahasiswa sekaligus. Periksa dengan hitungan tangan di slide: $w = 1$, $a = -5$.

In [ ]:
def prediksi(w, a, x=x):
    return ...

In [ ]:
cek(lambda: np.allclose(prediksi(1, -5, np.array([3., 5., 7.])), [0.1192, 0.5, 0.8808], atol=1e-4), 'Untuk x = 3, 5, 7 dengan w = 1, a = -5 seharusnya 0,119; 0,5; 0,881.')

### A3. Binary cross-entropy

$$L(w, a) = \frac{1}{n}\sum_{i=1}^{n} -\left[\,y_i \ln p_i + (1 - y_i)\ln(1 - p_i)\,\right]$$

`np.log` adalah ln. Baris `np.clip` sudah disediakan: ia menjaga `p` tidak tepat 0 atau 1, karena ln 0 tidak terdefinisi.

In [ ]:
def bce(w, a, x=x, y=y):
    n = len(x)
    p = np.clip(prediksi(w, a, x), 1e-12, 1 - 1e-12)
    return ...

In [ ]:
cek(lambda: dekat(bce(1, -5), 0.36500, 1e-3) and dekat(bce(0.3, -1.5), 0.46790, 1e-3) and dekat(bce(3, -15), 0.66088, 1e-3), 'Model A, B, C di slide: BCE 0,365; 0,468; 0,661.')

### A4. Bandingkan ketiga model dari slide

In [ ]:
for nama, w, a in [("A", 1, -5), ("B", 0.3, -1.5), ("C", 3, -15)]:
    print(f"Model {nama}: BCE = {bce(w, a):.3f}, akurasi = {akurasi(prediksi(w, a), y):.0%}")
plot_kurva(3, -15)

## Bagian B: gradien dan pelatihan

### B1. Gradien

Hasil penurunan di papan:

$$\frac{\partial L}{\partial w} = \frac{1}{n}\sum_{i=1}^{n} (p_i - y_i)\,x_i \qquad \frac{\partial L}{\partial a} = \frac{1}{n}\sum_{i=1}^{n} (p_i - y_i)$$

Bandingkan dengan fungsi `gradien` Pekan 2: hanya cara menghitung prediksinya yang berubah.

In [ ]:
def gradien(w, a, x=x, y=y):
    n = len(x)
    p = ...
    gw = ...
    ga = ...
    return gw, ga

In [ ]:
cek_gradien(gradien)

### B2. Loop gradient descent

Sama seperti Pekan 2. Simpan `(w, a, bce)` setelah setiap langkah.

In [ ]:
def gradient_descent(w, a, eta, langkah, x=x, y=y):
    riwayat = [(w, a, bce(w, a, x, y))]
    for t in range(langkah):
        ...
        ...
        riwayat.append((w, a, bce(w, a, x, y)))
    return riwayat

In [ ]:
cek(lambda: len(gradient_descent(0, 0, 0.3, 5)) == 6 and dekat(gradient_descent(0, 0, 0.3, 1)[1][0], 0.315) and dekat(gradient_descent(0, 0, 0.3, 1)[1][1], 0.0, 1e-9), 'Satu langkah dari (0, 0) dengan eta = 0,3 seharusnya memberi w = 0,315 dan a = 0.')

### B3. Latih dan lihat hasilnya

In [ ]:
riwayat = gradient_descent(0, 0, eta=0.3, langkah=3000)
w_akhir, a_akhir, L_akhir = riwayat[-1]
print(f"w = {w_akhir:.4f}, a = {a_akhir:.4f}, BCE = {L_akhir:.4f}")
print(f"decision boundary: {-a_akhir / w_akhir:.2f} jam, akurasi: {akurasi(prediksi(w_akhir, a_akhir), y):.0%}")
plot_loss(riwayat)
plot_kurva(w_akhir, a_akhir)

In [ ]:
cek(lambda: dekat(w_akhir, 0.8708, 1e-3) and dekat(a_akhir, -4.5718, 1e-3), 'Seharusnya w = 0,8708 dan a = -4,5718 (BCE 0,3592).')

## Bagian C: learning rate terbaik

**Arti terbaik:** BCE terkecil dengan budget **300 langkah**, mulai dari `w = 0, a = 0`. Lakukan sweep dengan skala log seperti Pekan 2, lalu perbesar di sekitar nilai terbaik.

In [ ]:
etas = [0.01, 0.03, 0.1, 0.3, 1, 3]
hasil = {}
for eta in etas:
    hasil[eta] = ...
plot_sweep(hasil)

In [ ]:
cek(lambda: sorted(hasil) == etas and all(len(r) == 301 for r in hasil.values()), 'Setiap eta harus dijalankan 300 langkah dari (0, 0).')

### C2. Perbesar di antara 0,3 dan 1, lalu simpan hasil terbaiknya di `eta_terbaik`

In [ ]:
etas_zoom = [...]
hasil_zoom = {}
for eta in etas_zoom:
    hasil_zoom[eta] = gradient_descent(0, 0, eta, 300)
eta_terbaik = plot_sweep(hasil_zoom)

In [ ]:
cek(lambda: 0.4 <= eta_terbaik <= 0.55, 'Untuk budget 300 langkah, eta terbaik ada sekitar 0,5.')

### C3. Lihat 50 langkah pertama

$\eta$ = 0,5 menang dengan budget 300 langkah. Bagaimana perjalanannya di awal?

In [ ]:
plot_loss({
    "eta = 0.3": gradient_descent(0, 0, 0.3, 50),
    "eta = 0.5": gradient_descent(0, 0, 0.5, 50),
    "eta = 1": gradient_descent(0, 0, 1, 50),
})

## Bagian D: dua fitur

Sekarang setiap mahasiswa punya dua angka: jam belajar `x1` dan jam tidur `x2`.

$$p_i = \sigma(w_1 x_{1i} + w_2 x_{2i} + a)$$

Gradiennya mengikuti pola yang sama, satu per parameter:

$$\frac{\partial L}{\partial w_1} = \frac{1}{n}\sum (p_i - y_i)\,x_{1i} \qquad \frac{\partial L}{\partial w_2} = \frac{1}{n}\sum (p_i - y_i)\,x_{2i} \qquad \frac{\partial L}{\partial a} = \frac{1}{n}\sum (p_i - y_i)$$

In [ ]:
def prediksi_2d(w1, w2, a):
    return ...


def bce_2d(w1, w2, a):
    n = len(y2)
    p = np.clip(prediksi_2d(w1, w2, a), 1e-12, 1 - 1e-12)
    return np.sum(-(y2 * np.log(p) + (1 - y2) * np.log(1 - p))) / n


def gradien_2d(w1, w2, a):
    n = len(y2)
    p = ...
    g1 = ...
    g2 = ...
    ga = ...
    return g1, g2, ga

In [ ]:
cek_gradien_2d(gradien_2d)

### D2. Loop untuk tiga parameter

In [ ]:
def gradient_descent_2d(eta, langkah):
    w1, w2, a = 0.0, 0.0, 0.0
    riwayat = [(w1, w2, a, bce_2d(w1, w2, a))]
    for t in range(langkah):
        ...
        ...
        riwayat.append((w1, w2, a, bce_2d(w1, w2, a)))
    return riwayat

In [ ]:
cek(lambda: np.allclose(gradient_descent_2d(0.3, 1)[1][:3], [0.168, 0.054375, -0.015]), 'Satu langkah dengan eta = 0,3 seharusnya memberi w1 = 0,168; w2 = 0,0544; a = -0,015.')

### D3. Latih dengan dua learning rate

Bandingkan $\eta$ = 0,1 dan $\eta$ = 0,3, masing-masing 3000 langkah. Mana yang lebih mulus, dan mana yang sampai lebih dekat?

In [ ]:
riwayat_01 = gradient_descent_2d(eta=0.1, langkah=3000)
riwayat_2d = gradient_descent_2d(eta=0.3, langkah=3000)
plot_loss({"eta = 0.1": riwayat_01, "eta = 0.3": riwayat_2d})
print(f"BCE akhir: eta 0.1 -> {riwayat_01[-1][-1]:.4f}, eta 0.3 -> {riwayat_2d[-1][-1]:.4f}")

w1_akhir, w2_akhir, a2_akhir, L2_akhir = riwayat_2d[-1]
print(f"eta = 0.3: w1 = {w1_akhir:.3f}, w2 = {w2_akhir:.3f}, a = {a2_akhir:.3f}, BCE = {L2_akhir:.4f}")
print(f"akurasi: {akurasi(prediksi_2d(w1_akhir, w2_akhir, a2_akhir), y2):.0%}")
plot_boundary_2d(w1_akhir, w2_akhir, a2_akhir)

In [ ]:
cek(lambda: dekat(w1_akhir, 0.8925, 1e-3) and dekat(w2_akhir, 0.9588, 1e-3) and dekat(a2_akhir, -11.0977, 1e-3), 'Seharusnya w1 = 0,893; w2 = 0,959; a = -11,098 (BCE 0,2836, akurasi 90%).')

## Bagian E: akurasi 95%

Data transaksi: 950 normal (y = 0) dan 50 penipuan (y = 1). Model malas menebak "normal" untuk semuanya. Jalankan saja.

In [ ]:
y_transaksi = np.array([0] * 950 + [1] * 50, dtype=float)
p_malas = np.zeros(1000)  # probabilitas penipuan selalu 0
print(f"akurasi model malas: {akurasi(p_malas, y_transaksi):.0%}")
print(f"penipuan yang tertangkap: {int(np.sum((p_malas >= 0.5) & (y_transaksi == 1)))} dari 50")

## Pertanyaan pemahaman (dikumpulkan)

Jawab masing-masing dalam dua sampai tiga kalimat.

1. Mengapa kita melatih dengan BCE, bukan langsung memaksimalkan akurasi?
2. Model di Bagian E akurasinya 95%. Bagus atau tidak? Apa yang sebaiknya dilaporkan selain akurasi?

*Jawabanmu:* ...